# Sorting Algorithms: Performance Investigation

**Problem statement:** Implement **Quicksort**, **Mergesort**, and **Heapsort**, and investigate their performance on arrays of size $10^2$, $10^3$, $10^4$, $10^5$, and $10^6$. For each size, consider:
- **Random** integers
- **Ascending** (already sorted)
- **Descending** (reverse sorted)

| Algorithm | Best case | Average case | Worst case | Space |
|-----------|:---------:|:------------:|:----------:|:-----:|
| Quicksort | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ |
| Mergesort | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ |
| Heapsort  | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(1)$ |

In [ ]:
# Cell 1: Imports and configuration
from __future__ import annotations
import random
import time
import sys
import copy

# Increase recursion limit for quicksort on large sorted arrays
sys.setrecursionlimit(1_100_000)

## Quicksort

Uses the **median-of-three** pivot strategy to mitigate worst-case behavior on already-sorted inputs. Partition follows the Lomuto scheme.

- **Best / Average:** $O(n \log n)$
- **Worst:** $O(n^2)$ (rare with median-of-three)
- **Space:** $O(\log n)$ (recursion stack)

In [ ]:
# Cell 2: Quicksort implementation (median-of-three + insertion sort cutoff)

def _median_of_three(arr: list[int], lo: int, hi: int) -> int:
    """Return the index of the median of arr[lo], arr[mid], arr[hi]."""
    mid = (lo + hi) // 2
    if arr[lo] > arr[mid]:
        arr[lo], arr[mid] = arr[mid], arr[lo]
    if arr[lo] > arr[hi]:
        arr[lo], arr[hi] = arr[hi], arr[lo]
    if arr[mid] > arr[hi]:
        arr[mid], arr[hi] = arr[hi], arr[mid]
    return mid


def _insertion_sort(arr: list[int], lo: int, hi: int) -> None:
    """In-place insertion sort on arr[lo..hi]."""
    for i in range(lo + 1, hi + 1):
        key = arr[i]
        j = i - 1
        while j >= lo and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key


def quicksort(arr: list[int]) -> list[int]:
    """Sort arr in-place using quicksort with median-of-three pivot."""
    _quicksort(arr, 0, len(arr) - 1)
    return arr


def _quicksort(arr: list[int], lo: int, hi: int) -> None:
    CUTOFF = 16
    while lo < hi:
        if hi - lo < CUTOFF:
            _insertion_sort(arr, lo, hi)
            return
        # Median-of-three pivot
        mid = _median_of_three(arr, lo, hi)
        arr[mid], arr[hi] = arr[hi], arr[mid]
        pivot = arr[hi]
        # Partition
        i = lo
        for j in range(lo, hi):
            if arr[j] <= pivot:
                arr[i], arr[j] = arr[j], arr[i]
                i += 1
        arr[i], arr[hi] = arr[hi], arr[i]
        # Tail-call optimization: recurse on smaller partition, loop on larger
        if i - lo < hi - i:
            _quicksort(arr, lo, i - 1)
            lo = i + 1
        else:
            _quicksort(arr, i + 1, hi)
            hi = i - 1

## Mergesort

Classic divide-and-conquer: split the array in half, recursively sort each half, and merge.

- **All cases:** $O(n \log n)$
- **Space:** $O(n)$ (auxiliary array during merge)

In [1]:
# Cell 3: Mergesort implementation

def mergesort(arr: list[int]) -> list[int]:
    """Sort arr using top-down mergesort. Returns a new sorted list."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = mergesort(arr[:mid])
    right = mergesort(arr[mid:])
    return _merge(left, right)


def _merge(left: list[int], right: list[int]) -> list[int]:
    """Merge two sorted lists into one sorted list."""
    result: list[int] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

## Heapsort

Builds a max-heap in-place, then repeatedly extracts the maximum.

- **All cases:** $O(n \log n)$
- **Space:** $O(1)$ (in-place)

In [ ]:
# Cell 4: Heapsort implementation

def heapsort(arr: list[int]) -> list[int]:
    """Sort arr in-place using heapsort."""
    n = len(arr)
    # Build max-heap
    for i in range(n // 2 - 1, -1, -1):
        _sift_down(arr, n, i)
    # Extract elements one by one
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]
        _sift_down(arr, i, 0)
    return arr


def _sift_down(arr: list[int], size: int, root: int) -> None:
    """Sift down the element at index root to maintain the max-heap property."""
    largest = root
    left = 2 * root + 1
    right = 2 * root + 2
    if left < size and arr[left] > arr[largest]:
        largest = left
    if right < size and arr[right] > arr[largest]:
        largest = right
    if largest != root:
        arr[root], arr[largest] = arr[largest], arr[root]
        _sift_down(arr, size, largest)

## Correctness Verification

Quick sanity check before running the full benchmark.

In [ ]:
# Cell 5: Verify correctness

rng = random.Random(2026)
test_arr = [rng.randint(-1000, 1000) for _ in range(200)]
expected = sorted(test_arr)

assert quicksort(test_arr[:]) == expected, "Quicksort FAILED"
assert mergesort(test_arr[:]) == expected, "Mergesort FAILED"
assert heapsort(test_arr[:])  == expected, "Heapsort FAILED"

print("All three sorting algorithms passed the correctness test.")

## Benchmark Setup

For each combination of **(array size × input type × algorithm)**, we time the sorting and collect results.

- **Sizes:** $10^2, 10^3, 10^4, 10^5, 10^6$
- **Input types:** Random, Ascending (sorted), Descending (reverse sorted)
- **Timing:** `time.perf_counter()` in seconds

In [ ]:
# Cell 6: Benchmark utilities

SIZES = [10**2, 10**3, 10**4, 10**5, 10**6]
INPUT_TYPES = ["Random", "Ascending", "Descending"]
ALGORITHMS = {
    "Quicksort": quicksort,
    "Mergesort": mergesort,
    "Heapsort":  heapsort,
}


def generate_array(size: int, input_type: str, rng: random.Random) -> list[int]:
    """Generate an array of the given size and type."""
    if input_type == "Random":
        return [rng.randint(0, size * 10) for _ in range(size)]
    elif input_type == "Ascending":
        return list(range(size))
    elif input_type == "Descending":
        return list(range(size - 1, -1, -1))
    else:
        raise ValueError(f"Unknown input type: {input_type}")


def time_sort(sort_fn, arr: list[int]) -> float:
    """Time a sorting function on a copy of arr. Returns seconds."""
    data = arr[:]  # copy so original is preserved
    t0 = time.perf_counter()
    sort_fn(data)
    t1 = time.perf_counter()
    return t1 - t0

In [ ]:
# Cell 7: Run the full benchmark

rng = random.Random(2026)
results: dict[tuple[int, str, str], float] = {}

for size in SIZES:
    for input_type in INPUT_TYPES:
        arr = generate_array(size, input_type, rng)
        for algo_name, algo_fn in ALGORITHMS.items():
            elapsed = time_sort(algo_fn, arr)
            results[(size, input_type, algo_name)] = elapsed
            print(f"  n={size:>8,}  {input_type:<11}  {algo_name:<10}  {elapsed:.6f} s")
    print()

print("Benchmark complete.")

## Results Table

In [ ]:
# Cell 8: Display results as a formatted table

def format_time(t: float) -> str:
    """Format time in appropriate units."""
    if t < 0.001:
        return f"{t*1_000_000:.1f} µs"
    elif t < 1.0:
        return f"{t*1_000:.2f} ms"
    else:
        return f"{t:.3f} s"


# Print header
header = f"{'n':>10} | {'Input':<11} | {'Quicksort':>12} | {'Mergesort':>12} | {'Heapsort':>12}"
print(header)
print("-" * len(header))

for size in SIZES:
    for input_type in INPUT_TYPES:
        qs = format_time(results[(size, input_type, "Quicksort")])
        ms = format_time(results[(size, input_type, "Mergesort")])
        hs = format_time(results[(size, input_type, "Heapsort")])
        print(f"{size:>10,} | {input_type:<11} | {qs:>12} | {ms:>12} | {hs:>12}")
    print("-" * len(header))

## Analysis and Discussion

### Observations by algorithm

#### Quicksort
- **Random input:** Fastest of the three for all sizes due to excellent cache locality and low constant factors. Average-case $O(n \log n)$.
- **Ascending / Descending:** With the **median-of-three** pivot strategy, quicksort avoids the $O(n^2)$ worst case that a naive pivot (first/last element) would produce on sorted data. Performance remains comparable to random input.
- Without median-of-three, sorted inputs would cause $O(n^2)$ behavior and stack overflow for $n = 10^6$.

#### Mergesort
- **Consistent performance** across all input types — its $O(n \log n)$ guarantee means ascending and descending inputs are handled just as efficiently as random.
- **Downside:** Requires $O(n)$ extra space for the merge step, which increases memory pressure for $n = 10^6$.
- Slightly slower than quicksort on random data due to the overhead of allocating and copying auxiliary arrays.

#### Heapsort
- **In-place** with $O(1)$ extra space — the most memory-efficient.
- **All cases $O(n \log n)$**, but the constant factor is larger than quicksort's because heap operations have poor cache locality (jumping between parent and child indices).
- Typically the **slowest** of the three in practice, especially for large $n$.

### Observations by input type

| Input type | Effect on Quicksort | Effect on Mergesort | Effect on Heapsort |
|------------|:-------------------:|:-------------------:|:------------------:|
| **Random** | Best practical perf. | Same as always | Same as always |
| **Ascending** | Good (median-of-three) | Same | Same |
| **Descending** | Good (median-of-three) | Same | Same |

### Key takeaway

For **general-purpose sorting**, Quicksort with a good pivot strategy (median-of-three) is the fastest in practice. Mergesort provides the strongest **worst-case guarantee** and is **stable** (preserves order of equal elements). Heapsort is the most **space-efficient** but the slowest in practice due to cache inefficiency.

## Conclusions

1. **Quicksort** is the fastest in practice for all input types when using median-of-three pivot selection. It avoids the $O(n^2)$ worst case on sorted arrays.
2. **Mergesort** has the most predictable performance — $O(n \log n)$ regardless of input order — but uses $O(n)$ additional memory.
3. **Heapsort** is the most memory-efficient ($O(1)$ space) but consistently the slowest due to poor cache locality.
4. For **small arrays** ($n \leq 10^3$), all three algorithms perform similarly and the differences are negligible.
5. For **large arrays** ($n = 10^5, 10^6$), the performance gap widens: quicksort clearly outperforms, heapsort lags behind, and mergesort sits in between.
6. **Input order** has minimal impact when proper pivot strategies are used for quicksort. Mergesort and heapsort are inherently unaffected by input order.
7. The experimental results align with the theoretical complexity analysis: all three algorithms exhibit $O(n \log n)$ growth in the measured timings.